## Model Simulation Settings
We want to simulate real-world foundation model + ABMIL models by selecting training hyperparameters from the following sampling space:
- 3 foundation models: Hoptimus1, Univ2, Virchow2
- Weight decay [0.0, 1e-04]
- Train-val split (stratified based on label)
    - No overlap in institution, overlap in institution
    - Dropout of some institutions -> 0-10, max. 10% of development data
- Random seed
- 70:30 train-val split

#### Fixed hyperparams:
* Train with ABMIL
* 5e-05 lr, AdamW + weight decay, cosine annealing learning rate, max epochs 30
* 10x magnification
* TCGA COADREAD FFPE diagnostic WSIs + as development set, * MSI status from NGS ([PanCancer 2018](https://pmc.ncbi.nlm.nih.gov/articles/PMC5352334/)) OR PCR([Nature 2012](https://www.nature.com/articles/nature11252))

#### Publicly available models:
##### HistoBistro CRC Transformer [Nature Wagner, 2023](https://pmc.ncbi.nlm.nih.gov/articles/PMC10507381) [Github](https://github.com/peng-lab/HistoBistro/tree/main/CancerCellCRCTransformer)
- ~13k CRC patients from 16 cohorts (including TCGA COADREAD) of resections + 2 biopsy cohorts (~1500 patients) for external testing
- Single cohort experiments: Train on one cohort + portion of cohort for in domain test, external test on all other cohorts
- Multi cohort experiments: Train on all except one resection cohorts, test on held-out cohort, test on external biopsy cohort
- CTransPath as feature extractor @ 512x512 px and 20x
- TransMIL, include stain augmentation with structure-preserving GAN; also tried ABMIL
- AdamW, 2*10^-5, 8 epochs, batch size 1
- ABMIL comparable performance: 0.96 AUC vs 0.97 TransMIL AUC


##### Niehus2023 [Cell2023](https://www.sciencedirect.com/science/article/pii/S2666379123000861?via%3Dihub) [Github](https://github.com/KatherLab/crc-models-2022)
- Train: QUASAR (~2,405) 5-fold cross-validation; Val: DACHS (~3,540)
- 512×512 px covering 256 µm -> 0.5 µm/px (~20×); resized to 224×224 px for network input (1.14 MPP)
- Tested 2 pre-trained feature encoders: Ciga2020, Wang2023 - **RetCCL2023** -> better one
- GatedABMIL
- Color jitter + stain normalization 
- Adam, 10-4 lr, batch size 1, 8 epochs
- Originally used Macenko stain normalization even on test, but also tested with out and found: "we observed an equivalent performance on noncolor-normalized tiles" without norm:  MSI 0.91 ± 0.01; with norm: 0.92 ± 0.01; "This provides further evidence that (1) the image-only WangattMIL (RetCCL2023) models generalize very well and do not suffer from domain shifts." 

### Define sampling space

In [1]:
import pandas as pd

fixed_hps = dict(
# Fixed hyperparams
lr = 5e-05,
opt = "AdamW",
scheduler = "cos",
max_epochs = 30,
mpp = 1,
earlystop_patience=5,
batch_size=1,
val_size=0.3,
max_data_dropout=0.1,

# Model architecture parameters
precompression_layer=True,
feature_size_comp=512,
feature_size_attn=256,
feature_size_comp_post=128,
p_dropout_fc=0.25,
p_dropout_atn=0.25
)

sampling_hps=dict(
# Sampling space
foundation_model = ["hoptimus1", "univ2", "virchow2"],
train_val_overlap_institution = [True, False],
train_val_dropout_institution = list(range(0,11)), # 0-10
weight_decay = [0, 1e-4],
)


In [ ]:
import random
from typing import Dict, List, Tuple, Optional
import numpy as np
import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
import os

def stratified_split(
    df: pd.DataFrame,
    label_cols: List[str],
    val_size: float,
    id_col: str,
    random_state: int = 25
) -> Dict[str, pd.DataFrame]:
    rng = np.random.RandomState(random_state)

    def stratified_sample(data: pd.DataFrame, frac: float) -> Tuple[pd.Index, pd.Index]:
        sample_ids = data[id_col].unique()
        n_test = round(len(sample_ids) * frac)

        # direct display of string-label tuples
        # print(
        #     f'{data["combined_labels"].unique()}, '
        #     f'frac={frac}, all={len(sample_ids)}, '
        #     f'ideal={frac*len(sample_ids):.2f}:{(1-frac)*len(sample_ids):.2f}, '
        #     f'split={n_test}:{len(sample_ids)-n_test}'
        # )

        test_ids = rng.choice(sample_ids, size=n_test, replace=False)
        test_idx = data[data[id_col].isin(test_ids)].index
        train_idx = data.index.difference(test_idx)
        return train_idx, test_idx

    patient_df = df[df["primary_patient_slide"]]
    assert patient_df["patient_id"].nunique() == len(patient_df)
    assert len(patient_df[patient_df[label_cols].isna().all(axis=1)]) == 0

    # keep string categories; tuple preserves multi-label combinations
    df["combined_labels"] = df[label_cols].apply(
        lambda row: tuple(row.dropna().astype(str)),
        axis=1
    )

    combined_label_combos = [
        lbl for lbl in df["combined_labels"].unique()
        if lbl != ()
    ]

    val_indices = pd.Index([])
    for combo in combined_label_combos:
        patient_ids = df.loc[df["combined_labels"] == combo, "patient_id"].unique()
        _, idx = stratified_sample(df[df["patient_id"].isin(patient_ids)], val_size)
        val_indices = val_indices.union(idx)

    train_indices = df.index.difference(val_indices)

    return {
        "train": df.loc[train_indices].drop(columns="combined_labels"),
        "val": df.loc[val_indices].drop(columns="combined_labels"),
        "full": df.drop(columns="combined_labels")
    }

def stratified_institutionwise_split(
    df: pd.DataFrame,
    label_col: str,
    val_size: float,
    id_col: str,
    random_state: int = 25,
    n_attempts: int = 2000
) -> Dict[str, pd.DataFrame]:
    """Institution-wise split with repeated randomized search to best match label distribution."""
    rng = np.random.RandomState(random_state)
    
    patient_df = df[df["primary_patient_slide"]]
    assert patient_df[id_col].nunique() == len(patient_df)
    assert len(patient_df[patient_df[label_col].isna()]) == 0

    institutions = df["site"].unique()
    inst_groups = df.groupby("site")
    inst_label_counts = {
        inst: inst_groups.get_group(inst)[label_col].value_counts().to_dict()
        for inst in institutions
    }

    label_totals = df[label_col].value_counts().to_dict()
    label_val_targets = {lbl: int(round(cnt * val_size)) for lbl, cnt in label_totals.items()}

    best_score = float("inf")
    best_split = None

    for _ in range(n_attempts):
        shuffled = institutions.copy()
        rng.shuffle(shuffled)

        val_insts, val_labels = [], {lbl: 0 for lbl in label_totals}

        for inst in shuffled:
            inst_counts = inst_label_counts[inst]
            fits = all(
                val_labels[lbl] + inst_counts.get(lbl, 0) <= label_val_targets[lbl]
                for lbl in inst_counts
            )
            if fits:
                val_insts.append(inst)
                for lbl in inst_counts:
                    val_labels[lbl] += inst_counts.get(lbl, 0)

        # compute score = sum(abs(diff))
        score = sum(abs(val_labels[lbl] - label_val_targets[lbl]) for lbl in label_totals)

        if score < best_score:
            best_score = score
            best_split = val_insts

    val_insts = best_split
    train_insts = [inst for inst in institutions if inst not in val_insts]

    train_idx = df[df["site"].isin(train_insts)].index
    val_idx = df[df["site"].isin(val_insts)].index
    
    return {
        "train": df.loc[train_idx],
        "val": df.loc[val_idx],
        "full": df
    }

def drop_sites(
    site_frac: Dict[str, float],
    n_drop_desired: int,
    max_fraction: float = 0.1,
    max_attempts: int = 1000,
    seed: Optional[int] = None
) -> Tuple[List[str], float]:
    rng = random.Random(seed)
    feasible_sites: List[str] = [s for s, frac in site_frac.items() if frac <= max_fraction]
    if len(feasible_sites) < n_drop_desired: return [], 0.0
    for _ in range(max_attempts):
        candidate_sites: List[str] = rng.sample(feasible_sites, n_drop_desired)
        total_frac: float = sum(site_frac[s] for s in candidate_sites)
        if total_frac <= max_fraction: return candidate_sites, total_frac
    return [], 0.0

seed=38
random.seed(seed)
n_simulations = 300 # 5-10min training * 60 = 300-600min / 2 GPUs = 150-300min ~ 2:30h-5:00h
full_hps = []
dev_df = pd.read_csv("../tcga_coadread.csv")
dev_df = dev_df[~dev_df["MSI"].isna()]
dev_df = dev_df[~dev_df["qc_excluded"]]
site_frac = dev_df.loc[dev_df["primary_patient_slide"], "site"].value_counts()/dev_df["patient_id"].nunique()
os.makedirs(f"./data/splits_n={n_simulations}", exist_ok=True)
for i in range(n_simulations):
    # Sample HPs
    sampled_hps = {}
    for k,v in sampling_hps.items():
        if k == "foundation_model": # stratified sampling
            sampled_hps[k] = v[i%len(v)]
        else:    
            sampled_hps[k] = random.choice(v)
    sampled_hps["seed"] = random.randint(1, 2**31 - 1)  # positive integer up to max 32-bit int


    # Split train val
    dropped_sites, total_frac = drop_sites(site_frac, n_drop_desired=sampled_hps["train_val_dropout_institution"], max_fraction=fixed_hps["max_data_dropout"], seed=random.seed(sampled_hps["seed"]))
    sampled_hps["dropped_sites"] = dropped_sites
    sampled_hps["dropped_data_frac"] = total_frac
    red_dev_df = dev_df[~dev_df["site"].isin(dropped_sites)]

    if sampled_hps["train_val_overlap_institution"]:
        print(f"Institutions can overlap, dropout={sampled_hps['train_val_dropout_institution']/(3.6*10**6)}kWh")
        splits = stratified_split(red_dev_df, label_cols=["MSI"], val_size=fixed_hps["val_size"], id_col="patient_id", random_state=sampled_hps["seed"])
    else:
        print(f"Institutions don't overlap, dropout={sampled_hps['train_val_dropout_institution']/(3.6*10**6)}kWh")
        splits = stratified_institutionwise_split(red_dev_df, label_col="MSI", val_size=fixed_hps["val_size"], id_col="patient_id", random_state=sampled_hps["seed"])
    train_df = splits["train"]
    val_df = splits["val"]
    print(f'Train: {splits["train"]["site"].nunique()} institutions, {train_df["patient_id"].nunique()} patients; \n'
          f'Val: {splits["val"]["site"].nunique()} institutions, {val_df["patient_id"].nunique()} patients; \n'
          f'Label distribution in full: {(red_dev_df.loc[red_dev_df["primary_patient_slide"], "MSI"].value_counts()/len(red_dev_df["primary_patient_slide"])).to_dict()}\n'
          f'Label distribution in train: {(train_df.loc[train_df["primary_patient_slide"], "MSI"].value_counts()/len(train_df["primary_patient_slide"])).to_dict()}\n'
          f'Label distribution in val: {(val_df.loc[val_df["primary_patient_slide"], "MSI"].value_counts()/len(val_df["primary_patient_slide"])).to_dict()}\n')

    
    train_df.to_csv(f"./data/splits_n={n_simulations}/train_{i}.csv", index=False)
    val_df.to_csv(f"./data/splits_n={n_simulations}/val_{i}.csv", index=False)

    hps = {**sampled_hps, **fixed_hps, "train_num_patients": train_df["patient_id"].nunique(), "val_num_patients": val_df["patient_id"].nunique()}
    full_hps.append(hps)

simulation_df = pd.DataFrame(full_hps)
simulation_df.to_csv(f"./data/simulation_hps_n={n_simulations}.csv", index=False)


Institutions don't overlap, dropout=6
Train: 11 institutions, 411 patients; 
Val: 20 institutions, 173 patients; 
Label distribution in full: {0: 0.8406779661016949, 1: 0.14915254237288136}
Label distribution in train: {0: 0.8450363196125908, 1: 0.15012106537530268}
Label distribution in val: {0: 0.8305084745762712, 1: 0.14689265536723164}

Institutions can overlap, dropout=9
Train: 26 institutions, 393 patients; 
Val: 21 institutions, 169 patients; 
Label distribution in full: {0: 0.8465608465608465, 1: 0.14462081128747795}
Label distribution in train: {0: 0.850632911392405, 1: 0.14430379746835442}
Label distribution in val: {0: 0.8372093023255814, 1: 0.14534883720930233}

Institutions can overlap, dropout=8
Train: 29 institutions, 383 patients; 
Val: 20 institutions, 164 patients; 
Label distribution in full: {0: 0.8414414414414414, 1: 0.14414414414414414}
Label distribution in train: {0: 0.8427835051546392, 1: 0.14432989690721648}
Label distribution in val: {0: 0.8383233532934131, 1